# Train an SPD of GPT-2 small on a Colab GPU

This notebook runs `scripts/train_spd.py` (Stochastic Parameter Decomposition of GPT-2 small's MLP down projections) for longer than a laptop can. The laptop run is in `docs/analyses/spd-baseline.md`. After 10,000 steps its subcomponents still had a cosine of 0.58–0.72 with their random start, so they cannot yet be read as the model's parts.

**Runtime.** *Runtime → Change runtime type → A100 GPU* (Colab Pro). A T4 is too slow.

**Before you start.** Upload `corpus.jsonl` from the repository root to `MyDrive/openinterp/corpus.jsonl`. The trainer leaves its documents out of the training data. The corpus is not in git because of its licence.

**Two runs.** Run each in its own folder (`RUN`):

1. `INIT = "random"`: the laptop run's settings, for 50,000 steps. Does the decomposition converge, and do its subcomponents then line up with neurons?
2. `INIT = "neurons"`: start with one subcomponent per neuron (V = I, U = W). Does SPD keep the neuron basis, or move away from it to a sparser one?

**If the runtime disconnects,** run every cell again with the same `RUN`. Training continues from the last checkpoint in Drive (every 1,000 steps), with a new optimizer.

**When it finishes,** send back `compare.json`, `train.log` and `layer_*.json` from the run's Drive folder. They are small. The `layer_*.safetensors` files are about 100 MB each; keep them in Drive.

In [ ]:
BRANCH = "spd-baseline"
RUN = "gpt2-spd-random-50k"  # the folder in MyDrive/openinterp; one per run
INIT = "random"  # "random", or "neurons" for one subcomponent per neuron
STEPS = 50_000
EXTRA = []  # more train_spd.py options, e.g. ["--batch-rows", "16"]

In [ ]:
import pathlib
import shlex

from google.colab import drive

drive.mount("/content/drive")
ROOT = pathlib.Path("/content/drive/MyDrive/openinterp")
OUT = ROOT / RUN
OUT.mkdir(parents=True, exist_ok=True)
CORPUS = ROOT / "corpus.jsonl"
assert CORPUS.exists(), f"Upload corpus.jsonl from the repository root to {CORPUS}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf openinterpretability
!git clone --depth 1 --branch {BRANCH} https://github.com/8W9aG/openinterpretability.git
%cd /content/openinterpretability
!pip install -q -e .

Training prints one line every 1,000 steps, with the hours left. Per layer, `l0` is the active subcomponents per token and `explained_variance` the fit of the CI-masked projection. On the laptop, 10,000 steps ended at 314–813 active subcomponents and a fit of 0.73–0.90.

In [ ]:
args = [
    "--layers", "all", "--steps", str(STEPS), "--eval-every", "1000",
    "--gate", "layer", "--d-gate", "512", "--init", INIT,
    "--coeff-imp", "6e-5", "--coeff-ci-output", "10", "--center", "--p-anneal", "0.1",
    "--imp-eps", "1e-3", "--gate-grad-clip", "1.0",
    "--scan-corpus", str(CORPUS), "--out", str(OUT), *EXTRA,
]  # fmt: skip
if (OUT / "checkpoint.pt").exists():
    args += ["--resume", str(OUT / "checkpoint.pt")]
train = shlex.join(["python", "train_spd.py", *args])
log = shlex.quote(str(OUT / "train.log"))
!cd scripts && {train} 2>&1 | tee -a {log}

Compare the result with the native neurons on held-out text: density at equal fit, alignment with the neuron axes, and drift from initialisation (see `scripts/spd_compare.py`).

In [ ]:
compare = shlex.join([
    "python", "spd_compare.py", "--checkpoint", str(OUT / "checkpoint.pt"),
    "--scan-corpus", str(CORPUS), "--out", str(OUT / "compare.json"),
])  # fmt: skip
!cd scripts && {compare}